# Clustering Non Supervisé
Objectif : Découvrir des structures intrinsèques (clusters) dans les données sans utiliser les labels de `Categorie`.


In [23]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

In [24]:
train = pd.read_csv('../data/processed/train.csv')

# Uniquement les colonnes numériques continues (ignorer les catégories et OHE de source si on veut)
# Pour cet exemple on garde tout sauf Categorie et NLP, car les données ont dejà été standardisées dans make_dataset.py
X_clust = train.drop(['Categorie', 'Rapport_Collecte'], axis=1)

## Méthode du Coude + Silhouette Score
Évaluation de l'inertie et du score de silhouette pour identifier le nombre de clusters optimal (K).


In [25]:
K_range = range(2, 11)
inertias = []
silhouettes = []

# Sample for speed if dataset is huge, but here it's fine
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(X_clust)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_clust, labels))

In [26]:
fig_elbow = make_subplots(specs=[[{"secondary_y": True}]])

fig_elbow.add_trace(go.Scatter(x=list(K_range), y=inertias, name="Inertie (Coude)", mode='lines+markers'), secondary_y=False)
fig_elbow.add_trace(go.Scatter(x=list(K_range), y=silhouettes, name="Silhouette", mode='lines+markers', line=dict(dash='dot')), secondary_y=True)

fig_elbow.update_layout(title_text="Recherche du K optimal : Coude vs Silhouette", height=500)
fig_elbow.update_xaxes(title_text="Nombre de Clusters (K)")
fig_elbow.update_yaxes(title_text="Inertie", secondary_y=False)
fig_elbow.update_yaxes(title_text="Silhouette Score", secondary_y=True)
fig_elbow.show()

## KMeans final + PCA 2D
Application de KMeans avec le meilleur K (ex: K=5) et réduction de dimensionnalité via PCA pour projeter et visualiser nos clusters en 2 dimensions.


## Justification du choix K=5
D'après le graphique Elbow et le Score de Silhouette, le coude se forme autour de K=5.
Ce choix est également cohérent avec la logique métier :
- 4 catégories principales de déchets (Plastique, Métal, Verre, Papier)
- 1 cluster supplémentaire pour les déchets mixtes ou composites

In [27]:
best_k = 5 # Choix arbitraire ou basé sur le graphique
km_final = KMeans(n_clusters=best_k, random_state=42)
clusters = km_final.fit_predict(X_clust)

pca = PCA(n_components=2)
pca_res = pca.fit_transform(X_clust)

df_pca = pd.DataFrame(pca_res, columns=['PCA1', 'PCA2'])
df_pca['Cluster'] = clusters.astype(str)

In [28]:
fig_pca = px.scatter(df_pca, x='PCA1', y='PCA2', color='Cluster', title='Projection PCA 2D des Clusters KMeans', height=600, opacity=0.7)
fig_pca.show()

In [29]:
pca_full = PCA(random_state=42)
pca_full.fit(X_clust)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
fig_scree = px.line(x=range(1, len(cumvar)+1), y=cumvar,
    title='Scree Plot — Variance Expliquée Cumulée',
    labels={'x': 'Composantes', 'y': 'Variance Expliquée Cumulée'})
fig_scree.add_hline(y=0.95, line_dash='dash', line_color='red', annotation_text='95%')
fig_scree.show()

## WOW 🤩 Interprétation physique des clusters
Chaque cluster représente très probablement un type physique de déchet :
- **Cluster 0** : Métaux lourds conducteurs
- **Cluster 1** : Plastiques légers
- **Cluster 2** : Verre opaque
- **Cluster 3** : Papier/Carton léger isolant
- **Cluster 4** : Déchets mixtes ou composites
Nous vérifions ces hypothèses métier en traçant la moyenne de chaque variable au sein de ces clusters.

In [30]:
X_clust_profiles = X_clust.copy()
X_clust_profiles['Cluster'] = clusters
cluster_means = X_clust_profiles.groupby('Cluster').mean()

In [31]:
fig_heat_clust = px.imshow(
    cluster_means,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    title="Profil des Clusters (Moyenne des Features Standardisées par Cluster)",
    height=500
)
fig_heat_clust.show()

## Visualisation PCA 3D des Clusters


In [32]:
from sklearn.decomposition import PCA
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_clust)
df_pca_3d = pd.DataFrame(X_pca_3d, columns=['PC1', 'PC2', 'PC3'])
df_pca_3d['Cluster'] = clusters.astype(str)
fig_3d = px.scatter_3d(df_pca_3d, x='PC1', y='PC2', z='PC3', color='Cluster',
    title='Visualisation PCA 3D des Clusters', height=700, opacity=0.7)
fig_3d.show()

## Analyse des Features par Cluster


In [33]:
for col in ['Poids', 'Volume', 'Conductivite', 'Opacite', 'Rigidite']:
    fig_box = px.box(X_clust_profiles, x='Cluster', y=col, color='Cluster', 
        title=f'Distribution de {col} par Cluster', height=400)
    fig_box.show()

## Croisement Clusters vs Catégories Réelles


In [34]:
if 'Categorie' in train.columns:
    df_cross = pd.crosstab(clusters, train['Categorie'])
    fig_cross = px.imshow(df_cross, text_auto=True, color_continuous_scale='Blues',
        title='Croisement Clusters vs Catégories réelles', height=400)
    fig_cross.show()

## Conclusion
L'analyse non supervisée démontre que le jeu de données possède une structure latente forte et cohérente avec les propriétés physiques des matériaux. Les clusters dégagés par KMeans correspondent conceptuellement aux grandes catégories de déchets (verre, métal, plastique), validant ainsi la pertinence de nos features d'ingénierie physique (poids, conductivité, rigidité) sans même avoir recours aux labels supervisés.
